# 🏈 Fourth Down Lab
## Football Offensive Efficiency Analytics

This notebook walks through the same portfolio project in a recruiter-friendly narrative: load the data, inspect it, answer football questions, and build an interpretable success model.

## 1. Load the active dataset
The notebook automatically prefers `cleaned_play_by_play.csv` (real data) and falls back to the included synthetic demo file.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
real = ROOT / 'data' / 'cleaned_play_by_play.csv'
demo = ROOT / 'data' / 'demo_play_by_play.csv'
source = real if real.exists() else demo
df = pd.read_csv(source)
print('Dataset:', source.name)
print('Rows:', f'{len(df):,}')
df.head()

## 2. Data quality check
Before modeling, verify the columns, missing values, and play types. This is an important interview talking point: models are only as good as the data going in.

In [ ]:
quality = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum(), 'unique': df.nunique()})
quality

## 3. Success rate by down

In [ ]:
by_down = df.groupby('down').agg(plays=('play_id','count'), success_rate=('success','mean'), avg_epa=('epa','mean')).reset_index()
by_down

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.bar(by_down['down'].astype(str), by_down['success_rate']*100)
ax.set(title='Offensive Success Rate by Down', xlabel='Down', ylabel='Success Rate (%)')
ax.grid(axis='y', alpha=.25)
plt.show()

## 4. Pass vs. run efficiency

In [ ]:
by_type = df.groupby('play_type').agg(plays=('play_id','count'), success_rate=('success','mean'), avg_epa=('epa','mean'), avg_yards=('yards_gained','mean')).round(3)
by_type

## 5. Third-down distance
This is a coaching-relevant slice that also demonstrates grouping and feature engineering.

In [ ]:
third = df[df['down'] == 3].copy()
third['distance_bucket'] = pd.cut(third['ydstogo'], [-1,3,6,10,float('inf')], labels=['Short (1-3)','Medium (4-6)','Long (7-10)','Very Long (11+)'])
third.groupby('distance_bucket', observed=True).agg(plays=('play_id','count'), success_rate=('success','mean'), avg_epa=('epa','mean')).round(3)

## 6. Baseline machine-learning model
Logistic regression is used because the target is binary and the model is easy to explain. The goal is not to claim the fanciest model; it is to establish a reproducible, interpretable baseline.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

model_df = df[['down','ydstogo','yardline_100','qtr','score_differential','play_type','shotgun','no_huddle','success']].dropna()
X = model_df.drop(columns='success')
y = model_df['success'].astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42, stratify=y)
num = ['down','ydstogo','yardline_100','qtr','score_differential','shotgun','no_huddle']
cat = ['play_type']
pre = ColumnTransformer([('num', StandardScaler(), num), ('cat', OneHotEncoder(handle_unknown='ignore'), cat)])
model = Pipeline([('preprocess', pre), ('model', LogisticRegression(max_iter=1000))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]
print('Accuracy:', round(accuracy_score(y_test,pred),3))
print('ROC AUC:', round(roc_auc_score(y_test,prob),3))

## 7. What I would improve next

- Compare logistic regression with random forest / gradient boosting using cross-validation.
- Add team and opponent strength controls.
- Add formation/personnel/motion when a source supports them.
- Calibrate probabilities before using them for decision recommendations.
- Deploy the dashboard and move the ingestion step to AWS for a cloud extension.

> If the notebook is using `demo_play_by_play.csv`, all numerical results are synthetic pipeline-validation results only. Run a real-data loader before presenting football findings.